# All 256 elementary cellular automata, and Lenia

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/cellular_automata)

A line of cells where each new cell depends on itself and its two neighbours: the three bits form a code $k \in \{0, \dots, 7\}$ and the rule number written in binary is the lookup table. All 256 rules run as one $(256, W)$ tensor advanced by a single gather: rule 90 draws Sierpinski's triangle, rule 30 looks random, rule 110 builds interacting structures and rule 184 moves traffic. A continuous cousin, Lenia, replaces the table by a smooth ring kernel and grows a gliding creature.

**Vector form**

$$x'_i = f_r(x_{i-1}, x_i, x_{i+1}),\qquad f_r(a, b, c) = \Big\lfloor \frac{r}{2^{4a + 2b + c}} \Big\rfloor \bmod 2,\qquad A' = \operatorname{clip}\Big(A + \tfrac1T\,G(K * A),\ 0,\ 1\Big)$$

**Index (tensor) form**

$$k_{ri} = P_{rij}\,w_j,\qquad x'_{ri} = T_{r,\,k_{ri}},\qquad U_{\mathbf p} = \sum_{\mathbf q} K_{\mathbf p - \mathbf q}\,A_{\mathbf q},\qquad G(U) = 2\,e^{-(U-\mu)^2/2\sigma^2} - 1$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_{rk}$ | `(rules[:, None] >> k) & 1` | (256, 8) | bit $k$ of rule number $r$: the new state when the neighbourhood code is $k$ |
| $P_{rij}$ | `torch.stack([roll(x, 1), x, roll(x, -1)], -1)` | (R, W, 3) | left, centre and right cell around cell $i$ on a ring |
| $k_{ri} = P_{rij} w_j$ | `einsum('rij,j->ri', P, [4, 2, 1])` | (R, W) | repeated $j$ ⇒ summed: the 3-bit neighbourhood code $4\,\text{left} + 2\,\text{centre} + \text{right}$ |
| $x'_{ri} = T_{r,k_{ri}}$ | `table.gather(1, code)` | (R, W) | one lookup advances every rule at once |
| $\tilde r,\ \bar r$ | `mirror_rule(r), complement_rule(r)` | (R,) | left-right mirror and 0/1 complement of a rule; the 256 rules form 88 symmetry classes |
| $K_{\mathbf p}$ | `lenia_kernel(S, R)` | (S, S) | smooth ring of radius $R = 13$, normalised to sum 1 |
| $U = K * A$ | `ifft2(fft2(A) * fft2(K)).real` | (S, S) | convolution on the periodic grid as a product of Fourier transforms |
| $A'$ | `lenia_step(A, K_hat, mu, sigma, T)` | (S, S) | grow where $U \approx \mu$, decay elsewhere, in steps of $1/T$ |

Wolfram sorted the elementary rules into four classes: (1) everything dies or fills, (2) still or periodic patterns, (3) chaos, such as rule 30, whose centre column was long used as a random number generator, and (4) complex, long-lived interacting structures, such as rule 110, which Matthew Cook proved (published 2004) is Turing complete. The mosaic colours each rule with a rough automatic test after 600 steps on a ring of 151 cells: uniform, periodic up to a shift, or no period found. The last group mixes classes 3 and 4, and telling them apart has no exact definition. Rule 90 adds neighbours mod 2, so from one cell it is Pascal's triangle mod 2, and rule 184 is the simplest traffic model: a car moves right if the cell ahead is free, so the number of cars is conserved. Unlike the Game of Life (2D, a $3\times3$ count, one rule), here every rule runs at once. Lenia (Bert Chan, 2019) makes space, time and states continuous; Orbium, its classic glider, starts from the cell table published with Lenia (MIT licence) and keeps its mass while it moves, an example of a soliton-like pattern that no conservation law enforces.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/cellular_automata/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors

FEATURED = {90: "single", 30: "single", 110: "random", 184: "random"}  # Sierpinski, chaos, complexity, traffic


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    w, n_steps = params["width"], params["generations"]
    rules = torch.arange(256, device=dev)
    table = tensors.rule_table(rules)  # (256, 8)

    # every rule from the same random line: one (T+1, 256, W) tensor
    mosaic = solve.run_eca(tensors.random_rows(256, 32, 0.5, 0, dev), table, 23)  # small thumbnails
    kind = solve.behaviour(tensors.random_rows(256, 151, 0.5, 1, dev), table, 600)  # (256,)
    reps = solve.equivalence_classes(rules)
    print(
        f"256 rules advanced together by one gather on the 3-cell code; {reps.unique().numel()} classes under symmetry"
    )
    names = ["uniform (class 1)", "periodic up to a shift (class 2)", "no period found (class 3 or 4)"]
    for c, name in enumerate(names):
        members = (kind == c).nonzero().flatten()
        print(f"{name:34s}: {len(members):3d} rules, {reps[members].unique().numel():2d} classes")
    print(f"rule 30 → {names[kind[30]]}, rule 90 → {names[kind[90]]}, rule 110 → {names[kind[110]]}")

    user = params["rule"]
    featured = {user: params["start"], **{r: s for r, s in FEATURED.items() if r != user}}
    featured = dict(list(featured.items())[:4])
    starts = {
        "single": tensors.single_cell(1, w, dev),
        "random": tensors.random_rows(1, w, params["density"], 2, dev),
    }
    x0 = torch.cat([starts[s] for s in featured.values()])  # (4, W)
    spacetime = solve.run_eca(x0, table[list(featured)], n_steps)  # (T+1, 4, W)
    for (r, s), cars in zip(featured.items(), spacetime.sum(-1).T):
        print(f"rule {r:3d} from a {s} start: live cells {int(cars[0])} → {int(cars[-1])}")

    a0 = tensors.orbium(params["lenia_size"], dev)
    kernel = tensors.lenia_kernel(params["lenia_size"], 13.0, dev)
    frames, mass = solve.run_lenia(a0, kernel, params["lenia_steps"])
    settled = mass[min(200, len(mass) // 2) :]
    print(
        f"Lenia Orbium: mass {mass[0]:.1f} → {mass[-1]:.1f}, "
        f"varying by {(settled.max() - settled.min()) / settled.mean():.2%} once settled"
    )
    shift, overlap = solve.best_shift(frames[len(frames) // 2], frames[-1])
    print(f"over the last {len(frames) // 2} steps the glider moved by {shift} cells, shape overlap {overlap:.3f}")
    return plot.animate(spacetime, list(featured), mosaic, kind, frames, mass, out_dir / "cellular_automata.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/cellular_automata/equation.py
r"""Rules → tensor translation.

Elementary cellular automata (Wolfram 1983): a ring of cells x_i ∈ {0, 1}; each new cell depends only on itself and
its two neighbours. The three cells form a 3-bit code

    k_{ri} = 4\,x_{r,i-1} + 2\,x_{r,i} + x_{r,i+1} = P_{rij}\,w_j,\qquad w = (4, 2, 1)   (j summed),

and the rule number r ∈ {0, …, 255} written in binary is the lookup table: bit k of r is the new state,

    x'_{ri} = T_{r,\,k_{ri}},\qquad T_{rk} = \lfloor r / 2^k \rfloor \bmod 2.

Putting all 256 rules in one (256, W) tensor, one `gather` advances every rule at once. Mirror symmetry
(left ↔ right) swaps codes k ↔ k̃ with the outer bits exchanged; complement symmetry (0 ↔ 1) maps
T_k → 1 − T_{7−k}. Together they cut 256 rules to 88 distinct behaviours.

Lenia (Chan 2019) is a continuous cellular automaton: states A ∈ [0, 1], a smooth ring kernel K of radius R
instead of the 3-cell neighbourhood, and a smooth growth function instead of a table,

    U = K * A,\qquad G(U) = 2\,e^{-(U-\mu)^2 / 2\sigma^2} - 1,\qquad A' = \operatorname{clip}(A + G(U)/T,\ 0,\ 1).

The convolution is a product of Fourier transforms on the periodic grid.
"""

import torch

W = torch.tensor([4, 2, 1])  # weights of left, centre, right in the neighbourhood code


def neighbourhood_code(x: torch.Tensor) -> torch.Tensor:
    r"""k_{ri} = P_{rij} w_j   (j summed over left, centre, right; periodic ring).   (R, W) → (R, W) int64"""
    patch = torch.stack([torch.roll(x, 1, -1), x, torch.roll(x, -1, -1)], dim=-1).long()  # (R, W, 3)
    return torch.einsum("rij,j->ri", patch, W.to(x.device))


def eca_step(x: torch.Tensor, table: torch.Tensor) -> torch.Tensor:
    r"""x'_{ri} = T_{r, k_{ri}}: one generation of every rule at once.   (R, W), (R, 8) → (R, W) uint8"""
    return table.gather(1, neighbourhood_code(x))


def mirror_rule(rules: torch.Tensor) -> torch.Tensor:
    """Rule number of the left-right mirror image: bit k̃ of the new rule = bit k of the old, k̃ = k with bits 0 and 2
    swapped.   (R,) → (R,)"""
    k = torch.arange(8)
    k_mirror = ((k & 1) << 2) | (k & 2) | ((k >> 2) & 1)  # (8,)
    bits = (rules[:, None] >> k_mirror) & 1  # (R, 8) bit k of the mirror = bit k̃ of the rule
    return (bits << k).sum(-1)


def complement_rule(rules: torch.Tensor) -> torch.Tensor:
    """Rule number with 0 and 1 exchanged: T'_k = 1 − T_{7−k}.   (R,) → (R,)"""
    k = torch.arange(8)
    bits = 1 - ((rules[:, None] >> (7 - k)) & 1)  # (R, 8)
    return (bits << k).sum(-1)


def growth(u: torch.Tensor, mu: float, sigma: float) -> torch.Tensor:
    r"""G(U) = 2 exp(−(U − μ)²/2σ²) − 1 ∈ [−1, 1]: grow where the neighbourhood is just right.   any shape"""
    return 2 * torch.exp(-((u - mu) ** 2) / (2 * sigma**2)) - 1


def lenia_step(a: torch.Tensor, k_hat: torch.Tensor, mu: float, sigma: float, t: float) -> torch.Tensor:
    r"""A' = clip(A + G(K * A)/T, 0, 1) with K * A = IFFT(FFT(A)·FFT(K)).   (..., S, S) → (..., S, S)"""
    u = torch.fft.ifft2(torch.fft.fft2(a) * k_hat).real
    return (a + growth(u, mu, sigma) / t).clamp(0, 1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/cellular_automata/tensors.py
"""Build the concrete tensors: the lookup table of every rule, starting lines, and the Lenia kernel and glider."""

import torch

# Orbium, the classic Lenia glider (R = 13, T = 10, μ = 0.15, σ = 0.015), as the 20 × 20 cell table distributed with
# B. W.-C. Chan's Lenia (github.com/Chakazul/Lenia, MIT licence). Values are cell states in [0, 1].
ORBIUM = [
    [0, 0, 0, 0, 0, 0, 0.1, 0.14, 0.1, 0, 0, 0.03, 0.03, 0, 0, 0.3, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0.08, 0.24, 0.3, 0.3, 0.18, 0.14, 0.15, 0.16, 0.15, 0.09, 0.2, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0.15, 0.34, 0.44, 0.46, 0.38, 0.18, 0.14, 0.11, 0.13, 0.19, 0.18, 0.45, 0, 0, 0],
    [0, 0, 0, 0, 0.06, 0.13, 0.39, 0.5, 0.5, 0.37, 0.06, 0, 0, 0, 0.02, 0.16, 0.68, 0, 0, 0],
    [0, 0, 0, 0.11, 0.17, 0.17, 0.33, 0.4, 0.38, 0.28, 0.14, 0, 0, 0, 0, 0, 0.18, 0.42, 0, 0],
    [0, 0, 0.09, 0.18, 0.13, 0.06, 0.08, 0.26, 0.32, 0.32, 0.27, 0, 0, 0, 0, 0, 0, 0.82, 0, 0],
    [0.27, 0, 0.16, 0.12, 0, 0, 0, 0.25, 0.38, 0.44, 0.45, 0.34, 0, 0, 0, 0, 0, 0.22, 0.17, 0],
    [0, 0.07, 0.2, 0.02, 0, 0, 0, 0.31, 0.48, 0.57, 0.6, 0.57, 0, 0, 0, 0, 0, 0, 0.49, 0],
    [0, 0.59, 0.19, 0, 0, 0, 0, 0.2, 0.57, 0.69, 0.76, 0.76, 0.49, 0, 0, 0, 0, 0, 0.36, 0],
    [0, 0.58, 0.19, 0, 0, 0, 0, 0, 0.67, 0.83, 0.9, 0.92, 0.87, 0.12, 0, 0, 0, 0, 0.22, 0.07],
    [0, 0, 0.46, 0, 0, 0, 0, 0, 0.7, 0.93, 1, 1, 1, 0.61, 0, 0, 0, 0, 0.18, 0.11],
    [0, 0, 0.82, 0, 0, 0, 0, 0, 0.47, 1, 1, 0.98, 1, 0.96, 0.27, 0, 0, 0, 0.19, 0.1],
    [0, 0, 0.46, 0, 0, 0, 0, 0, 0.25, 1, 1, 0.84, 0.92, 0.97, 0.54, 0.14, 0.04, 0.1, 0.21, 0.05],
    [0, 0, 0, 0.4, 0, 0, 0, 0, 0.09, 0.8, 1, 0.82, 0.8, 0.85, 0.63, 0.31, 0.18, 0.19, 0.2, 0.01],
    [0, 0, 0, 0.36, 0.1, 0, 0, 0, 0.05, 0.54, 0.86, 0.79, 0.74, 0.72, 0.6, 0.39, 0.28, 0.24, 0.13, 0],
    [0, 0, 0, 0.01, 0.3, 0.07, 0, 0, 0.08, 0.36, 0.64, 0.7, 0.64, 0.6, 0.51, 0.39, 0.29, 0.19, 0.04, 0],
    [0, 0, 0, 0, 0.1, 0.24, 0.14, 0.1, 0.15, 0.29, 0.45, 0.53, 0.52, 0.46, 0.4, 0.31, 0.21, 0.08, 0, 0],
    [0, 0, 0, 0, 0, 0.08, 0.21, 0.21, 0.22, 0.29, 0.36, 0.39, 0.37, 0.33, 0.26, 0.18, 0.09, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0.03, 0.13, 0.19, 0.22, 0.24, 0.24, 0.23, 0.18, 0.13, 0.05, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.06, 0.08, 0.09, 0.07, 0.05, 0.01, 0, 0, 0, 0, 0],
]


def rule_table(rules: torch.Tensor) -> torch.Tensor:
    """T_{rk} = bit k of the rule number r: the new state for neighbourhood code k = 4·left + 2·centre + right.
    (R,) → (R, 8) uint8."""
    k = torch.arange(8, device=rules.device)  # (8,)
    return ((rules[:, None] >> k) & 1).to(torch.uint8)


def single_cell(n_rows: int, width: int, device: torch.device) -> torch.Tensor:
    """One live cell in the middle of an empty line, for every rule: (R, W) uint8."""
    x = torch.zeros(n_rows, width, dtype=torch.uint8, device=device)
    x[:, width // 2] = 1
    return x


def random_rows(n_rows: int, width: int, density: float, seed: int, device: torch.device) -> torch.Tensor:
    """Random lines with a fraction `density` of live cells (fixed seed): (R, W) uint8."""
    gen = torch.Generator().manual_seed(seed)
    return (torch.rand(n_rows, width, generator=gen) < density).to(torch.uint8).to(device)


def lenia_kernel(size: int, radius: float, device: torch.device) -> torch.Tensor:
    r"""K(r) = exp(4 − 1/(ρ(1 − ρ))) on the ring 0 < ρ = r/R < 1, normalised to sum 1, centred on cell (0, 0) of a
    periodic (S, S) grid so that an FFT product is a circular convolution."""
    d = torch.arange(size, device=device, dtype=torch.float64)
    d = torch.minimum(d, size - d)  # periodic distance to cell 0
    rho = torch.sqrt(d[:, None] ** 2 + d[None, :] ** 2) / radius  # (S, S)
    inside = (rho > 0) & (rho < 1)
    shell = torch.exp(4 - 1 / (rho * (1 - rho)).clamp(min=1e-12))
    k = torch.where(inside, shell, torch.zeros_like(rho))
    return k / k.sum()


def orbium(size: int, device: torch.device) -> torch.Tensor:
    """An (S, S) world, zero except for the Orbium glider in the middle."""
    cells = torch.tensor(ORBIUM, dtype=torch.float64, device=device)
    a = torch.zeros(size, size, dtype=torch.float64, device=device)
    o = (size - cells.shape[0]) // 2
    a[o : o + cells.shape[0], o : o + cells.shape[1]] = cells
    return a


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/cellular_automata/solve.py
"""Run all 256 elementary rules side by side, sort them by behaviour, and follow a Lenia glider."""

import torch

from problems._common.progress import steps

from .equation import complement_rule, eca_step, lenia_step, mirror_rule

MU, SIGMA, T = 0.15, 0.015, 10.0  # Orbium's growth centre, width and time resolution (dt = 1/T)


def run_eca(x0: torch.Tensor, table: torch.Tensor, n_steps: int) -> torch.Tensor:
    """Space-time diagrams of every rule: (R, W), (R, 8) → (T+1, R, W) uint8."""
    hist = torch.empty((n_steps + 1, *x0.shape), dtype=torch.uint8, device=x0.device)
    hist[0] = x = x0
    for t in range(n_steps):
        hist[t + 1] = x = eca_step(x, table)
    return hist


def equivalence_classes(rules: torch.Tensor) -> torch.Tensor:
    """Smallest rule number among {r, mirror, complement, both}: the representative of r's class.   (R,) → (R,)"""
    m = mirror_rule(rules)
    family = torch.stack([rules, m, complement_rule(rules), complement_rule(m)])  # (4, R)
    return family.min(dim=0).values


def behaviour(x0: torch.Tensor, table: torch.Tensor, n_steps: int, max_period: int = 30) -> torch.Tensor:
    """A rough Wolfram classification after n_steps from random lines: 0 uniform (class 1), 1 periodic up to a shift
    (class 2), 2 no period found (class 3 or 4).   (R, W) → (R,)"""
    hist = run_eca(x0, table, n_steps)
    last = hist[-1]  # (R, W)
    uniform = (last == last[:, :1]).all(-1)
    w = last.shape[-1]
    shifts = torch.arange(-max_period, max_period + 1, device=x0.device)  # (S,)
    idx = (torch.arange(w, device=x0.device) - shifts[:, None]) % w  # (S, W) roll by s
    earlier = hist[-1 - max_period : -1]  # (P, R, W): states 1 … P steps before the end
    rolled = earlier[:, :, idx]  # (P, R, S, W)
    periodic = (rolled == last[None, :, None, :]).all(-1).any(-1).any(0)  # (R,)
    return torch.where(uniform, 0, torch.where(periodic, 1, 2))


def run_lenia(a0: torch.Tensor, kernel: torch.Tensor, n_steps: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Frames (T+1, S, S) float32 and total mass Σ A (T+1,) of a Lenia world."""
    k_hat = torch.fft.fft2(kernel)
    frames = torch.empty((n_steps + 1, *a0.shape), dtype=torch.float32, device=a0.device)
    mass = torch.empty(n_steps + 1, dtype=torch.float64, device=a0.device)
    a = a0
    for t in steps(n_steps + 1, "Lenia steps"):
        frames[t], mass[t] = a, a.sum()
        a = lenia_step(a, k_hat, MU, SIGMA, T)
    return frames, mass


def best_shift(a: torch.Tensor, b: torch.Tensor) -> tuple[tuple[int, int], float]:
    """Translation (dy, dx) that best maps a onto b on the periodic grid, and the normalised overlap ⟨a, b⟩ there."""
    a, b = a.double(), b.double()
    corr = torch.fft.ifft2(torch.fft.fft2(b) * torch.fft.fft2(a).conj()).real  # corr[s] = Σ_p a[p] b[p + s]
    s = corr.shape[-1]
    flat = int(corr.argmax())
    dy, dx = flat // s, flat % s
    dy, dx = (dy + s // 2) % s - s // 2, (dx + s // 2) % s - s // 2  # signed shifts
    return (dy, dx), (corr.max() / (a.norm() * b.norm())).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/cellular_automata/plot.py
"""Four famous rules growing row by row, a mosaic of all 256 rules, and the Lenia glider with its mass."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif

CMAP = ListedColormap(["#ffffff", "#1e3a8a"])
KIND_RGB = np.array([[156, 163, 175], [37, 99, 235], [220, 38, 38]]) / 255  # uniform, periodic, no period
LABEL = {30: "chaos", 90: "Sierpinski", 110: "complex", 184: "traffic"}


def mosaic_image(mosaic: torch.Tensor, kind: torch.Tensor) -> np.ndarray:
    """(T+1, 256, W) diagrams → one 16 × 16 RGB tile image, live cells coloured by behaviour class."""
    t, _, w = mosaic.shape
    tiles = mosaic.permute(1, 0, 2).cpu().numpy().reshape(16, 16, t, w)  # rule r = 16·row + column
    pad = np.zeros((16, 16, t + 2, w + 2))
    pad[:, :, 1:-1, 1:-1] = tiles
    img = np.ones((16 * (t + 2), 16 * (w + 2), 3))
    live = pad.transpose(0, 2, 1, 3).reshape(16 * (t + 2), 16 * (w + 2)) > 0
    colours = KIND_RGB[kind.cpu().numpy()].reshape(16, 16, 3)
    rgb = np.broadcast_to(colours[:, None, :, None, :], (16, t + 2, 16, w + 2, 3)).reshape(img.shape)
    img[live] = rgb[live]
    return img


def animate(
    spacetime: torch.Tensor,
    rules: list[int],
    mosaic: torch.Tensor,
    kind: torch.Tensor,
    frames: torch.Tensor,
    mass: torch.Tensor,
    path: Path,
) -> Path:
    """spacetime (T+1, 4, W), mosaic (T'+1, 256, W'), kind (256,), frames (L+1, S, S), mass (L+1,)."""
    st = spacetime.cpu().numpy().astype(float)
    n_rows = st.shape[0]
    lenia = frames.cpu().numpy()
    m = mass.cpu().numpy()

    fig = plt.figure(figsize=(10, 6.4))
    grid = fig.add_gridspec(2, 4, height_ratios=[1, 1.05], hspace=0.35, wspace=0.25)
    images = []
    for j, r in enumerate(rules):
        ax = fig.add_subplot(grid[0, j])
        im = ax.imshow(
            np.full_like(st[:, j], np.nan), cmap=CMAP, vmin=0, vmax=1, interpolation="nearest", aspect="auto"
        )
        ax.set_title(f"rule {r}" + (f" ({LABEL[r]})" if r in LABEL else ""), fontsize=10)
        ax.set_xticks([])
        ax.set_ylabel("time ↓" if j == 0 else "")
        ax.set_yticks([])
        images.append(im)

    ax_m = fig.add_subplot(grid[1, :2])
    ax_m.imshow(mosaic_image(mosaic, kind), interpolation="nearest")
    ax_m.set_xticks([])
    ax_m.set_yticks([])
    ax_m.set_title("all 256 rules (row-major, rule 0 top left)", fontsize=10)
    for c, name in enumerate(["uniform", "periodic", "no period found"]):
        ax_m.plot([], [], "s", color=KIND_RGB[c], label=name)
    ax_m.legend(fontsize=7, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3)

    ax_l = fig.add_subplot(grid[1, 2])
    lenia_im = ax_l.imshow(lenia[0], cmap="magma", vmin=0, vmax=1, interpolation="bilinear")
    ax_l.set_xticks([])
    ax_l.set_yticks([])
    lenia_title = ax_l.set_title("", fontsize=10)

    ax_c = fig.add_subplot(grid[1, 3])
    steps = np.arange(len(m))
    ax_c.plot(steps, m, color="#9ca3af", lw=1)
    (trace,) = ax_c.plot([], [], color="#dc2626", lw=1.6)
    ax_c.set_ylim(0, 1.3 * m.max())
    ax_c.set_xlabel("Lenia step")
    ax_c.set_title("mass Σ A (soliton: flat)", fontsize=10)

    def update(t: int):
        for j, im in enumerate(images):
            shown = st[:, j].copy()
            shown[t + 1 :] = np.nan  # rows not grown yet
            im.set_data(shown)
        s = round(t * (len(m) - 1) / max(1, n_rows - 1))
        lenia_im.set_data(lenia[s])
        lenia_title.set_text(f"Lenia Orbium, step {s}")
        trace.set_data(steps[: s + 1], m[: s + 1])

    return save_gif(fig, update, frame_indices(n_rows, 90), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'rule': 30,
    'start': 'single',
    'width': 161,
    'generations': 80,
    'density': 0.5,
    'lenia_size': 64,
    'lenia_steps': 600
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.cellular_automata")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.cellular_automata.main")
problem_dir = Path("problems/others/cellular_automata")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
